In [ ]:

# ============================================================
# 1-other.ipynb  ·  F1 原始变量采集（2018–2026）
# 采集五张原始变量表，供面板数据构建使用
#
# 表1  raw_race_results     — 正赛逐车结果（成绩、积分、完赛状态）
# 表2  raw_quali_laptimes   — 排位赛每车手最快圈（资源禀赋代理变量）
# 表3  raw_track_status     — 每圈赛道状态（SC/VSC 圈数原始数据）
# 表4  raw_event_meta       — 赛事静态元数据（赛道类型/长度/DRS 区数）
# 表5  raw_weather_summary  — 每场比赛天气摘要（雨战标记/温度/湿度/风速）
#
# ── 推荐运行顺序 ──────────────────────────────────────────────
#   Cell 1   （本单元）导入依赖库
#   Cell 2   配置参数 & CIRCUIT_META 硬编码常量
#   Cell 3   R session 合并采集：表1 + 表3 + 表5（2018–2025）
#   Cell 3b  2026 赛季补充：R session + Q session（进行中赛季）
#   Cell 4   Q session 排位赛：表2（2018–2025）
#   Cell 5   赛事元数据：表4（2018–2026，无需 session load）
#   Cell 8   合并各年 CSV → ALL_raw_*.csv（表1–3、5）
# ============================================================
import fastf1
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

print("fastf1 版本:", fastf1.__version__)


fastf1 版本: 3.8.1


In [2]:

# ============================================================
# Cell 2 — 配置参数 & 硬编码常量
# ============================================================

# ── 缓存与输出目录 ────────────────────────────────────────────
cache_dir  = os.path.join(os.getcwd(), "cache")
output_dir = os.path.join(os.getcwd(), "output")
os.makedirs(cache_dir,  exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

ALL_YEARS  = list(range(2018, 2027))
SKIP_YEARS = set()   # 变量采集较快，全量采，不跳过

# ── 赛道元数据字典 ─────────────────────────────────────────────
# key   = FastF1 schedule.Location 字段值（注意大小写与特殊字符）
# type  : 'street' = 街道/临时赛道, 'permanent' = 永久赛道
# length: 赛道长度 (km)
# drs   : DRS 区数量（以 2023 赛季版本为基准；部分赛道历年有调整）
CIRCUIT_META = {
    # ── 2018–2026 常设赛道 ──────────────────────────────────
    'Sakhir':            {'type': 'permanent', 'length': 5.412, 'drs': 2},
    'Jeddah':            {'type': 'street',    'length': 6.174, 'drs': 3},
    'Melbourne':         {'type': 'street',    'length': 5.278, 'drs': 4},  # 2022 重新设计后 DRS 区由 3 改为 4
    'Shanghai':          {'type': 'permanent', 'length': 5.451, 'drs': 2},
    'Miami':             {'type': 'street',    'length': 5.412, 'drs': 3},
    'Miami Gardens':     {'type': 'street',    'length': 5.412, 'drs': 3},  # fallback Location 名
    'Monaco':            {'type': 'street',    'length': 3.337, 'drs': 1},
    'Monte Carlo':       {'type': 'street',    'length': 3.337, 'drs': 1},  # fallback Location 名
    'Barcelona':         {'type': 'permanent', 'length': 4.675, 'drs': 2},
    'Montréal':          {'type': 'permanent', 'length': 4.361, 'drs': 2},
    'Montreal':          {'type': 'permanent', 'length': 4.361, 'drs': 2},  # fallback 无重音拼法
    'Spielberg':         {'type': 'permanent', 'length': 4.318, 'drs': 3},
    'Silverstone':       {'type': 'permanent', 'length': 5.891, 'drs': 2},
    'Budapest':          {'type': 'permanent', 'length': 4.381, 'drs': 1},
    'Spa-Francorchamps': {'type': 'permanent', 'length': 7.004, 'drs': 2},
    'Spa':               {'type': 'permanent', 'length': 7.004, 'drs': 2},  # fallback 短名
    'Zandvoort':         {'type': 'permanent', 'length': 4.259, 'drs': 2},
    'Monza':             {'type': 'permanent', 'length': 5.793, 'drs': 2},
    'Baku':              {'type': 'street',    'length': 6.003, 'drs': 2},
    'Singapore':         {'type': 'street',    'length': 4.940, 'drs': 3},
    'Marina Bay':        {'type': 'street',    'length': 4.940, 'drs': 3},  # fallback Location 名
    'Suzuka':            {'type': 'permanent', 'length': 5.807, 'drs': 1},
    'Lusail':            {'type': 'permanent', 'length': 5.380, 'drs': 2},  # 卡塔尔 2023+
    'Losail':            {'type': 'permanent', 'length': 5.380, 'drs': 2},  # 卡塔尔 2021 旧拼法
    'Austin':            {'type': 'permanent', 'length': 5.513, 'drs': 2},
    'Mexico City':       {'type': 'permanent', 'length': 4.304, 'drs': 3},
    'São Paulo':         {'type': 'permanent', 'length': 4.309, 'drs': 2},
    'Sao Paulo':         {'type': 'permanent', 'length': 4.309, 'drs': 2},  # fallback 无变音符
    'Interlagos':        {'type': 'permanent', 'length': 4.309, 'drs': 2},  # fallback 赛道名
    'Las Vegas':         {'type': 'street',    'length': 6.201, 'drs': 2},
    'Abu Dhabi':         {'type': 'permanent', 'length': 5.281, 'drs': 2},
    'Yas Island':        {'type': 'permanent', 'length': 5.281, 'drs': 2},  # fallback Location 名
    'Yas Marina':        {'type': 'permanent', 'length': 5.281, 'drs': 2},  # fallback 赛道名
    'Madrid':            {'type': 'street',    'length': 5.474, 'drs': 3},  # 西班牙/马德里 GP 2026+
    # ── 历史/一次性赛道 ──────────────────────────────────────
    'Hockenheim':        {'type': 'permanent', 'length': 4.574, 'drs': 2},  # 德国 GP 2018–2019
    'Le Castellet':      {'type': 'permanent', 'length': 5.842, 'drs': 2},  # 法国 GP 2018–2022
    'Mugello':           {'type': 'permanent', 'length': 5.245, 'drs': 1},  # 托斯卡纳 GP 2020
    'Nürburg':           {'type': 'permanent', 'length': 5.148, 'drs': 1},  # 艾菲尔 GP 2020
    'Nürburgring':       {'type': 'permanent', 'length': 5.148, 'drs': 1},  # fallback 全名
    'Portimão':          {'type': 'permanent', 'length': 4.653, 'drs': 2},  # 葡萄牙 GP 2020–2021
    'Istanbul':          {'type': 'permanent', 'length': 5.338, 'drs': 1},  # 土耳其 GP 2020–2021
    'Sochi':             {'type': 'street',    'length': 5.848, 'drs': 2},  # 俄罗斯 GP 2018–2021
    'Imola':             {'type': 'permanent', 'length': 4.909, 'drs': 1},  # 艾米利亚罗马涅 GP 2020–2024
}

print(f"配置完成：{len(ALL_YEARS)} 赛季, CIRCUIT_META 含 {len(CIRCUIT_META)} 条目")
print(f"缓存目录: {cache_dir}")
print(f"输出目录: {output_dir}")


配置完成：9 赛季, CIRCUIT_META 含 43 条目
缓存目录: /Users/jiangheng/Desktop/Work/Courses/Management Methods 2/term paper/cache
输出目录: /Users/jiangheng/Desktop/Work/Courses/Management Methods 2/term paper/output


In [ ]:
# ============================================================
# Cell 3 — R session 合并采集（表1 raw_race_results
#                              + 表3 raw_track_status
#                              + 表5 raw_weather_summary）
#          范围: 2018–2025（2026 赛季由下方单独 Cell 处理）
#
#   优化：每场比赛只 load 一次 R session（laps=True, weather=True）
#         相比分三个 Cell 各自 load，API 请求次数降为 1/3
# ============================================================

def _wmean(wd, col):
    """从 weather_data 取某列均值；列不存在时返回 NaN。"""
    return round(float(wd[col].mean()), 3) if col in wd.columns else np.nan

for year in ALL_YEARS:
    if year in SKIP_YEARS or year == 2026:
        print(f"⊙ 跳过 {year}")
        continue

    try:
        schedule    = fastf1.get_event_schedule(year, include_testing=False)
        race_rounds = schedule.loc[schedule['RoundNumber'] > 0, 'RoundNumber'].tolist()
    except Exception as e:
        print(f"⚠ {year} 赛历获取失败: {e}")
        continue

    print(f"\n── {year} 赛季: {len(race_rounds)} 站 ──")
    results_rows = []   # 表1: DataFrame list
    status_rows  = []   # 表3: DataFrame list
    weather_rows = []   # 表5: dict list

    for rn in race_rounds:
        try:
            sess = fastf1.get_session(year, rn, 'R')
            # ── 单次 load：laps + weather 同时拉取 ──────────────────
            sess.load(laps=True, telemetry=False, weather=True, messages=False)
            event_name = sess.event['EventName']

            # ── 表1: 正赛结果 ────────────────────────────────────────
            want_cols = ['DriverNumber', 'Abbreviation', 'TeamName',
                         'GridPosition', 'Position', 'Points', 'Status']
            res = sess.results[[c for c in want_cols if c in sess.results.columns]].copy()
            res.rename(columns={'Abbreviation': 'DriverAbbr'}, inplace=True)
            res.insert(0, 'Year', year)
            res.insert(1, 'Round', rn)
            res.insert(2, 'EventName', event_name)
            classified = (
                sess.laps
                .groupby('Driver')['LapNumber'].max()
                .reset_index()
                .rename(columns={'Driver': 'DriverAbbr', 'LapNumber': 'ClassifiedLaps'})
            )
            res = res.merge(classified, on='DriverAbbr', how='left')
            res['TotalRaceLaps'] = getattr(sess, 'total_laps', np.nan)
            results_rows.append(res)

            # ── 表3: 赛道状态 ────────────────────────────────────────
            st = (
                sess.laps[['LapNumber', 'TrackStatus']]
                .drop_duplicates()
                .dropna(subset=['LapNumber'])
                .copy()
            )
            st.insert(0, 'Year', year)
            st.insert(1, 'Round', rn)
            st.insert(2, 'EventName', event_name)
            status_rows.append(st[['Year', 'Round', 'EventName', 'LapNumber', 'TrackStatus']])

            # ── 表5: 天气摘要 ────────────────────────────────────────
            wd = sess.weather_data
            if wd is None or len(wd) == 0:
                is_rainy = np.nan
                weather_rows.append({
                    'Year': year, 'Round': rn, 'EventName': event_name,
                    'IsRainy': np.nan, 'RainyLapFraction': np.nan,
                    'AirTemp_mean': np.nan, 'TrackTemp_mean': np.nan,
                    'Humidity_mean': np.nan, 'WindSpeed_mean': np.nan,
                })
            else:
                is_rainy   = bool(wd['Rainfall'].any()) if 'Rainfall' in wd.columns else np.nan
                rainy_frac = round(float(wd['Rainfall'].mean()), 4) if 'Rainfall' in wd.columns else np.nan
                weather_rows.append({
                    'Year': year, 'Round': rn, 'EventName': event_name,
                    'IsRainy': is_rainy, 'RainyLapFraction': rainy_frac,
                    'AirTemp_mean':   _wmean(wd, 'AirTemp'),
                    'TrackTemp_mean': _wmean(wd, 'TrackTemp'),
                    'Humidity_mean':  _wmean(wd, 'Humidity'),
                    'WindSpeed_mean': _wmean(wd, 'WindSpeed'),
                })

            print(f"  ✓ R{rn:02d} {event_name}  "
                  f"({len(res)} 车手 | {len(st)} 圈状态行 | IsRainy={is_rainy})")
        except Exception as e:
            print(f"  ✗ {year} R{rn:02d} 失败: {e}")

    # ── 逐年三张表存盘 ──────────────────────────────────────────
    for df_rows, tname in [
        (results_rows, 'raw_race_results'),
        (status_rows,  'raw_track_status'),
    ]:
        if df_rows:
            yr_df = pd.concat(df_rows, ignore_index=True)
            path  = os.path.join(output_dir, f"{year}_{tname}.csv")
            yr_df.to_csv(path, index=False, encoding='utf-8-sig')
            print(f"  → 存盘: {os.path.basename(path)} ({len(yr_df)} 行)")

    if weather_rows:
        yr_df = pd.DataFrame(weather_rows)
        path  = os.path.join(output_dir, f"{year}_raw_weather_summary.csv")
        yr_df.to_csv(path, index=False, encoding='utf-8-sig')
        print(f"  → 存盘: {os.path.basename(path)} ({len(yr_df)} 行)")


In [ ]:

# ============================================================
# Cell 3b — 2026 赛季补充采集（进行中赛季，独立于 Cell 3/4 处理）
#
#   说明：2026 赛季尚在进行中，故与 Cell 3/4 的 2018–2025 批量循环
#         分开处理，避免因赛历不完整导致错误。
#         · R session → 表1 raw_race_results
#                       表3 raw_track_status
#                       表5 raw_weather_summary
#         · Q session → 表2 raw_quali_laptimes
#
#   MAX_2026_ROUNDS：控制已完成站数上限；
#                    赛季结束后可设为完整站数（约 24）再重跑。
# ============================================================

YEAR_2026       = 2026
MAX_2026_ROUNDS = 2   # 只采集前 N 站，后续赛季结束后可调大

def _wmean(wd, col):
    """从 weather_data 取某列均值；列不存在时返回 NaN。"""
    return round(float(wd[col].mean()), 3) if col in wd.columns else np.nan

try:
    schedule_2026    = fastf1.get_event_schedule(YEAR_2026, include_testing=False)
    race_rounds_2026 = schedule_2026.loc[schedule_2026['RoundNumber'] > 0, 'RoundNumber'].tolist()
    race_rounds_2026 = race_rounds_2026[:MAX_2026_ROUNDS]
    print(f"2026 赛季：采集前 {MAX_2026_ROUNDS} 站（Round {race_rounds_2026}）")
except Exception as e:
    print(f"⚠ 2026 赛历获取失败: {e}")
    race_rounds_2026 = []

# ── R session：表1 + 表3 + 表5 ───────────────────────────────
results_rows, status_rows, weather_rows = [], [], []

for rn in race_rounds_2026:
    try:
        sess = fastf1.get_session(YEAR_2026, rn, 'R')
        sess.load(laps=True, telemetry=False, weather=True, messages=False)
        event_name = sess.event['EventName']

        # 表1
        want_cols = ['DriverNumber', 'Abbreviation', 'TeamName',
                     'GridPosition', 'Position', 'Points', 'Status']
        res = sess.results[[c for c in want_cols if c in sess.results.columns]].copy()
        res.rename(columns={'Abbreviation': 'DriverAbbr'}, inplace=True)
        res.insert(0, 'Year', YEAR_2026); res.insert(1, 'Round', rn); res.insert(2, 'EventName', event_name)
        classified = (sess.laps.groupby('Driver')['LapNumber'].max().reset_index()
                      .rename(columns={'Driver': 'DriverAbbr', 'LapNumber': 'ClassifiedLaps'}))
        res = res.merge(classified, on='DriverAbbr', how='left')
        res['TotalRaceLaps'] = getattr(sess, 'total_laps', np.nan)
        results_rows.append(res)

        # 表3
        st = (sess.laps[['LapNumber', 'TrackStatus']].drop_duplicates().dropna(subset=['LapNumber']).copy())
        st.insert(0, 'Year', YEAR_2026); st.insert(1, 'Round', rn); st.insert(2, 'EventName', event_name)
        status_rows.append(st[['Year', 'Round', 'EventName', 'LapNumber', 'TrackStatus']])

        # 表5
        wd = sess.weather_data
        if wd is None or len(wd) == 0:
            is_rainy = np.nan
            weather_rows.append({'Year': YEAR_2026, 'Round': rn, 'EventName': event_name,
                                  'IsRainy': np.nan, 'RainyLapFraction': np.nan,
                                  'AirTemp_mean': np.nan, 'TrackTemp_mean': np.nan,
                                  'Humidity_mean': np.nan, 'WindSpeed_mean': np.nan})
        else:
            is_rainy   = bool(wd['Rainfall'].any()) if 'Rainfall' in wd.columns else np.nan
            rainy_frac = round(float(wd['Rainfall'].mean()), 4) if 'Rainfall' in wd.columns else np.nan
            weather_rows.append({'Year': YEAR_2026, 'Round': rn, 'EventName': event_name,
                                  'IsRainy': is_rainy, 'RainyLapFraction': rainy_frac,
                                  'AirTemp_mean':   _wmean(wd, 'AirTemp'),
                                  'TrackTemp_mean': _wmean(wd, 'TrackTemp'),
                                  'Humidity_mean':  _wmean(wd, 'Humidity'),
                                  'WindSpeed_mean': _wmean(wd, 'WindSpeed')})

        print(f"  ✓ R{rn:02d} {event_name}  ({len(res)} 车手 | IsRainy={is_rainy})")
    except Exception as e:
        print(f"  ✗ 2026 R{rn:02d} 失败: {e}")

# 存盘
for df_rows, tname in [(results_rows, 'raw_race_results'), (status_rows, 'raw_track_status')]:
    if df_rows:
        yr_df = pd.concat(df_rows, ignore_index=True)
        path  = os.path.join(output_dir, f"{YEAR_2026}_{tname}.csv")
        yr_df.to_csv(path, index=False, encoding='utf-8-sig')
        print(f"  → 存盘: {os.path.basename(path)} ({len(yr_df)} 行)")
if weather_rows:
    yr_df = pd.DataFrame(weather_rows)
    path  = os.path.join(output_dir, f"{YEAR_2026}_raw_weather_summary.csv")
    yr_df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"  → 存盘: {os.path.basename(path)} ({len(yr_df)} 行)")

# ── Q session：表2 排位赛圈时 ────────────────────────────────
quali_rows = []
for rn in race_rounds_2026:
    try:
        sess = fastf1.get_session(YEAR_2026, rn, 'Q')
        sess.load(laps=True, telemetry=False, weather=False, messages=False)
        event_name = sess.event['EventName']
        laps_valid = sess.laps[['Driver', 'Team', 'LapTime']].dropna(subset=['LapTime'])
        best = (laps_valid.groupby(['Driver', 'Team'])['LapTime'].min().reset_index()
                .rename(columns={'Driver': 'DriverAbbr'}))
        best['BestLapTime_s'] = best['LapTime'].dt.total_seconds()
        best = best.drop(columns=['LapTime'])
        best.insert(0, 'Year', YEAR_2026); best.insert(1, 'Round', rn); best.insert(2, 'EventName', event_name)
        quali_rows.append(best)
        print(f"  ✓ Q{rn:02d} {event_name} ({len(best)} 车手)")
    except Exception as e:
        print(f"  ✗ 2026 Q{rn:02d} 失败: {e}")

if quali_rows:
    yr_df = pd.concat(quali_rows, ignore_index=True)
    path  = os.path.join(output_dir, f"{YEAR_2026}_raw_quali_laptimes.csv")
    yr_df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"  → 存盘: {os.path.basename(path)} ({len(yr_df)} 行)")


In [ ]:
# ============================================================
# Cell 4 — 表2: 排位赛每车手最快圈 (raw_quali_laptimes)
#          范围: 2018–2025（2026 赛季由上方单独 Cell 处理）
#          Session: 'Q'  load: laps=True
#          Sprint 赛周亦使用 'Q'（主赛资格赛，非冲刺资格赛 'SQ'）
#          输出字段: Year, Round, EventName, DriverAbbr, Team, BestLapTime_s
#          BestLapTime_s 单位：秒，下游用于计算 quali_gap = BestLapTime_s - pole_time
# ============================================================

for year in ALL_YEARS:
    if year in SKIP_YEARS or year == 2026:
        print(f"⊙ 跳过 {year}")
        continue

    try:
        schedule    = fastf1.get_event_schedule(year, include_testing=False)
        race_rounds = schedule.loc[schedule['RoundNumber'] > 0, 'RoundNumber'].tolist()
    except Exception as e:
        print(f"⚠ {year} 赛历获取失败: {e}")
        continue

    print(f"\n── {year} 赛季: {len(race_rounds)} 站 ──")
    year_rows = []

    for rn in race_rounds:
        try:
            sess = fastf1.get_session(year, rn, 'Q')
            sess.load(laps=True, telemetry=False, weather=False, messages=False)
            event_name = sess.event['EventName']

            # ── 每车手最快圈：去掉 LapTime=NaT，按车手取最小圈时 ──────
            laps_valid = sess.laps[['Driver', 'Team', 'LapTime']].dropna(subset=['LapTime'])
            best = (
                laps_valid
                .groupby(['Driver', 'Team'])['LapTime']
                .min()
                .reset_index()
                .rename(columns={'Driver': 'DriverAbbr'})
            )
            # Timedelta → float 秒
            best['BestLapTime_s'] = best['LapTime'].dt.total_seconds()
            best = best.drop(columns=['LapTime'])
            best.insert(0, 'Year', year)
            best.insert(1, 'Round', rn)
            best.insert(2, 'EventName', event_name)

            year_rows.append(best)
            print(f"  ✓ R{rn:02d} {event_name} ({len(best)} 车手)")
        except Exception as e:
            print(f"  ✗ {year} R{rn:02d} 失败: {e}")

    if year_rows:
        yr_df = pd.concat(year_rows, ignore_index=True)
        path  = os.path.join(output_dir, f"{year}_raw_quali_laptimes.csv")
        yr_df.to_csv(path, index=False, encoding='utf-8-sig')
        print(f"  → 存盘: {os.path.basename(path)} ({len(yr_df)} 行)")


In [3]:
# ============================================================
# Cell 5 — 表4: 赛事静态元数据 (raw_event_meta)
#          范围: 2018–2026（赛历无需 session load，2026 全年赛历一并采集）
#          无需 session load；使用 get_event_schedule + CIRCUIT_META 字典
#          输出字段: Year, Round, EventName, Location, Country,
#                    EventFormat, CircuitType, CircuitLength_km, DRS_Zones
#          注意: 若某 Location 不在 CIRCUIT_META 中，末尾会列出 unmapped 条目
#          ALL_raw_event_meta.csv 在本 Cell 直接产出（无需最终合并 Cell 再处理）
# ============================================================

all_rows = []
unmapped = set()

for year in ALL_YEARS:
    try:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        rounds   = schedule[schedule['RoundNumber'] > 0]
    except Exception as e:
        print(f"⚠ {year} 赛历获取失败: {e}")
        continue

    for _, row in rounds.iterrows():
        loc  = row.get('Location', '')
        meta = CIRCUIT_META.get(loc)
        if meta is None:
            unmapped.add(loc)

        all_rows.append({
            'Year':             year,
            'Round':            int(row['RoundNumber']),
            'EventName':        row.get('EventName', ''),
            'Location':         loc,
            'Country':          row.get('Country', ''),
            'EventFormat':      row.get('EventFormat', ''),
            'CircuitType':      meta['type']   if meta else np.nan,
            'CircuitLength_km': meta['length'] if meta else np.nan,
            'DRS_Zones':        meta['drs']    if meta else np.nan,
        })

meta_df = pd.DataFrame(all_rows)

# ── 逐年分表存盘 ─────────────────────────────────────────────
for year in ALL_YEARS:
    yr = meta_df[meta_df['Year'] == year]
    if not yr.empty:
        path = os.path.join(output_dir, f"{year}_raw_event_meta.csv")
        yr.to_csv(path, index=False, encoding='utf-8-sig')

# ── 全量合并直接存盘 ─────────────────────────────────────────
all_path = os.path.join(output_dir, "ALL_raw_event_meta.csv")
meta_df.to_csv(all_path, index=False, encoding='utf-8-sig')
years = sorted(meta_df['Year'].dropna().unique().astype(int))
print(f"已存盘: ALL_raw_event_meta.csv ({len(meta_df)} 行, 赛季: {years})")

# ── 未映射赛道提示 ──────────────────────────────────────────
if unmapped:
    print(f"\n⚠ 以下 Location 未在 CIRCUIT_META 中找到（CircuitType/Length/DRS 为 NaN）：")
    for loc in sorted(unmapped):
        print(f"   '{loc}'")
    print("  → 请手动补充至 Cell 2 的 CIRCUIT_META 字典后重新运行本 Cell")
else:
    print("✓ 全部 Location 均已映射")


已存盘: ALL_raw_event_meta.csv (195 行, 赛季: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)])
✓ 全部 Location 均已映射


In [ ]:

# ============================================================
# Cell 8 — 合并各年 CSV → ALL_raw_*.csv
#
#   注意：raw_event_meta 已在 Cell 5 直接产出 ALL_raw_event_meta.csv，
#         本 Cell 仅合并其余四张表（race_results / quali_laptimes /
#         track_status / weather_summary）。
#
#   逻辑：glob 匹配 output/ 下所有 {year}_{tname}.csv，
#         pd.concat 后存盘为 ALL_{tname}.csv。
# ============================================================

TABLE_NAMES = [
    'raw_race_results',
    'raw_quali_laptimes',
    'raw_track_status',
    'raw_weather_summary',
]

for tname in TABLE_NAMES:
    pattern = os.path.join(output_dir, f"*_{tname}.csv")
    files   = sorted(glob.glob(pattern))
    if not files:
        print(f"  ⚠ 无文件匹配: *_{tname}.csv，跳过")
        continue

    parts    = [pd.read_csv(f, encoding='utf-8-sig') for f in files]
    combined = pd.concat(parts, ignore_index=True)
    out_path = os.path.join(output_dir, f"ALL_{tname}.csv")
    combined.to_csv(out_path, index=False, encoding='utf-8-sig')

    years = sorted(combined['Year'].dropna().unique().astype(int)) if 'Year' in combined.columns else []
    print(f"  已合并: ALL_{tname}.csv  ({len(combined)} 行, 赛季: {years})")

print(f"\n输出目录: {output_dir}")
print("\n── 全部五张表 ALL_raw_*.csv 一览 ──")
for fname in sorted(glob.glob(os.path.join(output_dir, "ALL_raw_*.csv"))):
    rows = len(pd.read_csv(fname, encoding='utf-8-sig'))
    print(f"  {os.path.basename(fname):45s}  {rows} 行")


  已合并: ALL_raw_race_results.csv  (3502 行, 赛季: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)])
  已合并: ALL_raw_quali_laptimes.csv  (3474 行, 赛季: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)])
  已合并: ALL_raw_track_status.csv  (12036 行, 赛季: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)])
  已合并: ALL_raw_weather_summary.csv  (175 行, 赛季: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)])

输出目录: /Users/jiangheng/Desktop/Work/Courses/Management Methods 2/term paper/output

── 全部五张表 ALL_raw_*.csv 一览 ──
  ALL_raw_event_meta.csv                         195 行
  ALL_raw_quali_laptimes.csv                     3474 行
  